In [8]:
with open("policies.jsonl", "r") as f:
    lines = f.readlines()

lines

['{"doc_id": "about_1", "source_file": "about.docx", "category": "О компании", "doc_type": "policy", "title": null, "content": "ИНТЕРНЕТ-МАГАЗИН OSTORE.KG O!Store – интернет-магазин, у которого более 110 филиалов по всему Кыргызстану. В магазинах огромный выбор смартфонов мировых брендов. Есть всё, для подключения к скоростному мобильному интернету: роутеры, винглы, семейные комплекты для дома! Больше 20 000 аксессуаров для мобильных устройств: кабели, переходники, адаптеры, акустические колонки и наушники, ультрамодные сумки, бананки и рюкзаки, портативные зарядки, устройства для системы «Умный дом» и многое другое. Только сертифицированный товар от официальных поставщиков. Гарантийное обслуживание. Смартфон можно купить В КРЕДИТ или В РАССРОЧКУ без процентов и переплаты. Постоянные Акции, Скидки и Специальные предложения. Бесплатная доставка по Бишкеку. На нашем сайте вы найдёте детальное описание каждого товара БОНУС при заказе смартфона онлайн! Оперативная обратная связь по телефон

In [9]:
import json

data = [json.loads(line) for line in lines]
data

[{'doc_id': 'about_1',
  'source_file': 'about.docx',
  'category': 'О компании',
  'doc_type': 'policy',
  'title': None,
  'content': 'ИНТЕРНЕТ-МАГАЗИН OSTORE.KG O!Store – интернет-магазин, у которого более 110 филиалов по всему Кыргызстану. В магазинах огромный выбор смартфонов мировых брендов. Есть всё, для подключения к скоростному мобильному интернету: роутеры, винглы, семейные комплекты для дома! Больше 20 000 аксессуаров для мобильных устройств: кабели, переходники, адаптеры, акустические колонки и наушники, ультрамодные сумки, бананки и рюкзаки, портативные зарядки, устройства для системы «Умный дом» и многое другое. Только сертифицированный товар от официальных поставщиков. Гарантийное обслуживание. Смартфон можно купить В КРЕДИТ или В РАССРОЧКУ без процентов и переплаты. Постоянные Акции, Скидки и Специальные предложения. Бесплатная доставка по Бишкеку. На нашем сайте вы найдёте детальное описание каждого товара БОНУС при заказе смартфона онлайн! Оперативная обратная связь п

In [10]:
import re

MAX_LEN = 500
OVERLAP_WORDS = 10

for item in data:
    text = f"{item.get('title', '')}. {item['content']}".strip()

    if len(text) <= MAX_LEN:
        item["chunks"] = [text]
        continue

    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current = ""

    for sent in sentences:

        if len(current) + len(sent) + 1 <= MAX_LEN:
            current += " " + sent

        else:
            chunks.append(current.strip())

            overlap = " ".join(
                current.split()[-OVERLAP_WORDS:]
            )

            current = overlap + " " + sent

    if current:
        chunks.append(current.strip())

    item["chunks"] = chunks

data

[{'doc_id': 'about_1',
  'source_file': 'about.docx',
  'category': 'О компании',
  'doc_type': 'policy',
  'title': None,
  'content': 'ИНТЕРНЕТ-МАГАЗИН OSTORE.KG O!Store – интернет-магазин, у которого более 110 филиалов по всему Кыргызстану. В магазинах огромный выбор смартфонов мировых брендов. Есть всё, для подключения к скоростному мобильному интернету: роутеры, винглы, семейные комплекты для дома! Больше 20 000 аксессуаров для мобильных устройств: кабели, переходники, адаптеры, акустические колонки и наушники, ультрамодные сумки, бананки и рюкзаки, портативные зарядки, устройства для системы «Умный дом» и многое другое. Только сертифицированный товар от официальных поставщиков. Гарантийное обслуживание. Смартфон можно купить В КРЕДИТ или В РАССРОЧКУ без процентов и переплаты. Постоянные Акции, Скидки и Специальные предложения. Бесплатная доставка по Бишкеку. На нашем сайте вы найдёте детальное описание каждого товара БОНУС при заказе смартфона онлайн! Оперативная обратная связь п

In [12]:
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

tokenizer = AutoTokenizer.from_pretrained("google/embeddinggemma-300m")
model = SentenceTransformer("google/embeddinggemma-300m")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

In [4]:
tokenizer.encode("Hello world")

[2, 9259, 1902, 1]

In [36]:
from collections import Counter

for item in data:
    dense_vectors = []
    for chunk in item["chunks"]:
        dense_embedding = model.encode_document(chunk)
        dense_vectors.append(dense_embedding)

    item["dense_vectors"] = dense_vectors
    counter = Counter(tokenizer.encode(item["content"]))
    item["sparse_indices"] = list(counter.keys())
    item["sparse_values"] = list(counter.values())

In [37]:
data

[{'doc_id': 'about_1',
  'source_file': 'about.docx',
  'category': 'О компании',
  'doc_type': 'policy',
  'title': None,
  'content': 'ИНТЕРНЕТ-МАГАЗИН OSTORE.KG O!Store – интернет-магазин, у которого более 110 филиалов по всему Кыргызстану. В магазинах огромный выбор смартфонов мировых брендов. Есть всё, для подключения к скоростному мобильному интернету: роутеры, винглы, семейные комплекты для дома! Больше 20 000 аксессуаров для мобильных устройств: кабели, переходники, адаптеры, акустические колонки и наушники, ультрамодные сумки, бананки и рюкзаки, портативные зарядки, устройства для системы «Умный дом» и многое другое. Только сертифицированный товар от официальных поставщиков. Гарантийное обслуживание. Смартфон можно купить В КРЕДИТ или В РАССРОЧКУ без процентов и переплаты. Постоянные Акции, Скидки и Специальные предложения. Бесплатная доставка по Бишкеку. На нашем сайте вы найдёте детальное описание каждого товара БОНУС при заказе смартфона онлайн! Оперативная обратная связь п

In [39]:
from qdrant_client import QdrantClient, models
import os
from dotenv import load_dotenv
load_dotenv()

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

collection_name = "o_policies_pro"

# Create a collection
if client.collection_exists(collection_name):
    print(f"Collection '{collection_name}' already exists.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense": models.VectorParams(
            # Size of an individual token vector
            size=768,
            # Distance function for token similarity
            distance=models.Distance.DOT,
            # Enable multi-vector mode with MaxSim
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            # Disable HNSW indexing
            hnsw_config=models.HnswConfigDiff(m=0),
        ),
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams()
        }
    )
    print(f"Collection '{collection_name}' created.")

client.create_payload_index(
    collection_name=collection_name,
    field_name="doc_id",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="source_file",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="category",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="title",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="doc_type",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

Collection 'o_policies_pro' created.


UpdateResult(operation_id=10, status=<UpdateStatus.COMPLETED: 'completed'>)

In [38]:
client.delete_collection(collection_name)

True

In [40]:
points = []
for i, item in enumerate(data):
    point = models.PointStruct(
        id=i,
        vector={"dense": item["dense_vectors"], 
            "sparse": models.SparseVector(
                indices=item["sparse_indices"],
                values=item["sparse_values"],
            )
        },
        payload={
            "doc_id": item["doc_id"],
            "source_file": item["source_file"],
            "category": item["category"],
            "title": item.get("title", ""),
            "doc_type": item["doc_type"],
            "content": item["content"],
        }
            
    )

    points.append(point)

client.upsert(
    collection_name=collection_name,
    points=points,
)

UpdateResult(operation_id=11, status=<UpdateStatus.COMPLETED: 'completed'>)

In [44]:
query = "Скидки на продукцию Apple"
dense_vector = model.encode(query)
sparse_counter = Counter(tokenizer.encode(query))
indices = list(sparse_counter.keys())
values = list(sparse_counter.values())
sparse_vector = models.SparseVector(indices=indices, values=values)

search_result = client.query_points(
    collection_name=collection_name,
    prefetch=[
        models.Prefetch(
            query=dense_vector,
            using="dense",
            limit=5,
        ),
        models.Prefetch(
            query=sparse_vector,
            using="sparse",
            limit=5,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=3,
)

In [45]:
search_result

QueryResponse(points=[ScoredPoint(id=0, version=11, score=0.5, payload={'doc_id': 'about_1', 'source_file': 'about.docx', 'category': 'О компании', 'title': None, 'doc_type': 'policy', 'content': 'ИНТЕРНЕТ-МАГАЗИН OSTORE.KG O!Store – интернет-магазин, у которого более 110 филиалов по всему Кыргызстану. В магазинах огромный выбор смартфонов мировых брендов. Есть всё, для подключения к скоростному мобильному интернету: роутеры, винглы, семейные комплекты для дома! Больше 20 000 аксессуаров для мобильных устройств: кабели, переходники, адаптеры, акустические колонки и наушники, ультрамодные сумки, бананки и рюкзаки, портативные зарядки, устройства для системы «Умный дом» и многое другое. Только сертифицированный товар от официальных поставщиков. Гарантийное обслуживание. Смартфон можно купить В КРЕДИТ или В РАССРОЧКУ без процентов и переплаты. Постоянные Акции, Скидки и Специальные предложения. Бесплатная доставка по Бишкеку. На нашем сайте вы найдёте детальное описание каждого товара БОН